# Stress Prediction V35 — Domain Feature + SVR Stacking

V34(ExtraTrees tree-quantile + pair-neighbor 블렌딩, Public MAE 0.127187)에서
아래 4가지를 추가/변경했습니다.

1. `ExtraTreesRegressor(criterion="absolute_error")` — 평가지표(MAE)와 트리 분할 기준을 일치시킴
2. 도메인 지식 기반 피처 추가 — 혈압 단계(대한고혈압학회 기준), 혈당/콜레스테롤 임상 구간, 알로스타틱 부하(Allostatic Load) 지표
3. Pair-neighbor 예측을 1-NN(argmin) → k-NN(k=5) 평균으로 변경해 이상치 민감도 완화
4. SVR을 세 번째 모델로 추가하고, 세 모델(Tree / Pair-kNN / SVR)의 OOF 예측을 RidgeCV로 스태킹해 가중치를 자동 산출 (수동 quantile/weight 튜닝을 대체)

⚠️ 데이터 파일 경로(`DATA_DIR`)는 실제 환경에 맞게 수정해서 실행해주세요.
저는 실제 train.csv/test.csv에 접근할 수 없어 이 노트북을 직접 실행/검증하지 못했습니다.
컬럼명이 실제 데이터와 다르면 `PAIR_COLUMNS`, `add_domain_features` 부분을 맞춰 수정해야 합니다.


In [1]:

from __future__ import annotations

import os
from itertools import combinations
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold


TARGET = "stress_score"
ID_COLUMN = "ID"
RANDOM_SEED = 42
N_ESTIMATORS = 800  # absolute_error는 계산량이 커서 1200 -> 800으로 축소 (필요시 조정)
PAIR_QUANTILE = 0.48
PAIR_K = 5  # 기존 argmin(1개) -> k=5 최근접 평균으로 변경

FEATURE_COPY_COUNTS = {
    "mean_working": 1,
    "bmi": 4,
    "cholesterol": 2,
    "height": 2,
    "glucose": 3,
    "weight": 2,
    "cholesterol_glucose_ratio": 2,
    "bone_density": 4,
}

PAIR_COLUMNS = [
    "mean_working",
    "bmi",
    "cholesterol",
    "height",
    "glucose",
    "weight",
    "cholesterol_glucose_ratio",
    "bone_density",
]

# Allostatic load에 사용할 생체지표 (train stats로만 z-score 계산)
ALLOSTATIC_COLUMNS = [
    "systolic_blood_pressure",
    "diastolic_blood_pressure",
    "cholesterol",
    "glucose",
    "bmi",
    "bone_density",
]


In [2]:

def add_features(frame: pd.DataFrame) -> pd.DataFrame:
    """다른 행이나 Test 전체 통계를 사용하지 않는 행 단위 파생변수 (V34와 동일)."""
    result = frame.copy()
    height_m = result["height"] / 100.0
    sbp = result["systolic_blood_pressure"]
    dbp = result["diastolic_blood_pressure"]
    result["bmi"] = result["weight"] / height_m.pow(2)
    result["pulse_pressure"] = sbp - dbp
    result["mean_arterial_pressure"] = (sbp + 2.0 * dbp) / 3.0
    result["blood_pressure_ratio"] = sbp / (dbp + 1e-6)
    result["cholesterol_glucose_ratio"] = result["cholesterol"] / (
        result["glucose"] + 1e-6
    )
    result["missing_count"] = result.isna().sum(axis=1)
    return result


def _bp_stage(sbp: float, dbp: float) -> str:
    """대한고혈압학회 기준 혈압 단계 분류."""
    if pd.isna(sbp) or pd.isna(dbp):
        return "unknown"
    if sbp >= 140 or dbp >= 90:
        return "stage2"
    if sbp >= 130 or dbp >= 80:
        return "stage1"
    if sbp >= 120:
        return "elevated"
    return "normal"


def _glucose_stage(glucose: float) -> str:
    """ADA 공복혈당 기준 (mg/dL 가정) — 단위가 다르면 임계값 조정 필요."""
    if pd.isna(glucose):
        return "unknown"
    if glucose >= 126:
        return "diabetes_range"
    if glucose >= 100:
        return "prediabetes_range"
    return "normal"


def _cholesterol_stage(chol: float) -> str:
    """ATP III 총콜레스테롤 기준 (mg/dL 가정)."""
    if pd.isna(chol):
        return "unknown"
    if chol >= 240:
        return "high"
    if chol >= 200:
        return "borderline"
    return "desirable"


def add_domain_features(frame: pd.DataFrame) -> pd.DataFrame:
    """임상 기준 기반 범주형 피처 (행 단위, leakage 없음)."""
    result = frame.copy()
    result["bp_stage"] = [
        _bp_stage(s, d)
        for s, d in zip(result["systolic_blood_pressure"], result["diastolic_blood_pressure"])
    ]
    result["glucose_stage"] = result["glucose"].apply(_glucose_stage)
    result["cholesterol_stage"] = result["cholesterol"].apply(_cholesterol_stage)
    return result


def compute_train_stats(train_frame: pd.DataFrame, columns: list[str]) -> dict:
    """Allostatic load 계산용 train 평균/표준편차. 반드시 train fold에서만 호출."""
    stats = {}
    for col in columns:
        values = train_frame[col].to_numpy(dtype=float)
        stats[col] = (np.nanmean(values), np.nanstd(values) + 1e-6)
    return stats


def add_allostatic_load(frame: pd.DataFrame, train_stats: dict) -> pd.DataFrame:
    """train_stats(해당 fold의 train 통계)만 사용해 z-score 이탈 개수를 합산."""
    result = frame.copy()
    abnormal_flags = pd.DataFrame(index=frame.index)
    for col, (mean, std) in train_stats.items():
        z = (frame[col] - mean) / std
        abnormal_flags[f"{col}_abnormal"] = (z.abs() > 1).astype(int)
    result["allostatic_load"] = abnormal_flags.sum(axis=1)
    return result


def add_feature_copies(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.copy()
    for feature, additional_copies in FEATURE_COPY_COUNTS.items():
        for copy_index in range(1, additional_copies + 1):
            result[f"{feature}__copy{copy_index}"] = result[feature]
    return result


def build_fold_features(train_raw: pd.DataFrame, other_raw: pd.DataFrame):
    """train fold 통계만으로 train/other(val 또는 test)에 동일하게 피처를 적용."""
    train_feat = add_domain_features(add_features(train_raw))
    other_feat = add_domain_features(add_features(other_raw))
    stats = compute_train_stats(train_feat, ALLOSTATIC_COLUMNS)
    train_feat = add_allostatic_load(train_feat, stats)
    other_feat = add_allostatic_load(other_feat, stats)
    return train_feat, other_feat


In [3]:

def make_preprocessor(frame: pd.DataFrame) -> ColumnTransformer:
    categorical = frame.select_dtypes(
        include=["object", "category", "string"]
    ).columns.tolist()
    numerical = frame.select_dtypes(include=["number"]).columns.tolist()
    return ColumnTransformer(
        [
            (
                "categorical",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
                        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
                    ]
                ),
                categorical,
            ),
            (
                "numerical",
                SimpleImputer(strategy="median", add_indicator=True),
                numerical,
            ),
        ]
    )


def tree_prediction(train_features, val_features, target) -> np.ndarray:
    """criterion=absolute_error 로 MAE에 직접 최적화. quantile-of-trees 트릭은 제거하고
    표준 예측(트리 평균)을 사용 — absolute_error 자체가 이미 MAE 최적화이므로 더 견고함."""
    weighted_train = add_feature_copies(train_features)
    weighted_val = add_feature_copies(val_features)
    pipeline = Pipeline(
        [
            ("preprocessor", make_preprocessor(weighted_train)),
            (
                "model",
                ExtraTreesRegressor(
                    n_estimators=N_ESTIMATORS,
                    min_samples_leaf=1,
                    max_features=1,
                    criterion="absolute_error",
                    random_state=RANDOM_SEED,
                    n_jobs=-1,
                ),
            ),
        ]
    )
    pipeline.fit(weighted_train, target)
    return pipeline.predict(weighted_val)


def empirical_rank_transform(train_values: np.ndarray, test_values: np.ndarray):
    """Train 분포만으로 Train과 Test 값을 변환 (V34와 동일)."""
    median = np.nanmedian(train_values)
    train_values = np.nan_to_num(train_values, nan=median)
    test_values = np.nan_to_num(test_values, nan=median)
    sorted_train = np.sort(train_values)
    train_rank = np.searchsorted(sorted_train, train_values, side="right") / len(train_values)
    test_rank = np.searchsorted(sorted_train, test_values, side="right") / len(train_values)
    return train_rank.astype(np.float32), test_rank.astype(np.float32)


def pair_knn_prediction(train_features, val_features, target) -> np.ndarray:
    """각 2차원 피처쌍에서 k=5 최근접 train target 평균 -> 피처쌍 간 quantile 집계.
    V34는 argmin 1개만 사용해 이상치에 취약했음."""
    train_rank, val_rank = {}, {}
    for column in PAIR_COLUMNS:
        train_rank[column], val_rank[column] = empirical_rank_transform(
            train_features[column].to_numpy(dtype=float),
            val_features[column].to_numpy(dtype=float),
        )

    target_values = target.to_numpy(dtype=float)
    k = min(PAIR_K, len(target_values))
    neighbor_targets = []
    for first, second in combinations(PAIR_COLUMNS, 2):
        distance = (
            np.abs(val_rank[first][:, None] - train_rank[first][None, :])
            + np.abs(val_rank[second][:, None] - train_rank[second][None, :])
        )
        nearest_idx = np.argpartition(distance, kth=k - 1, axis=1)[:, :k]
        neighbor_mean = target_values[nearest_idx].mean(axis=1)
        neighbor_targets.append(neighbor_mean)
    return np.quantile(np.column_stack(neighbor_targets), PAIR_QUANTILE, axis=1)


def svr_prediction(train_features, val_features, target) -> np.ndarray:
    """트리/거리 기반 모델과 다른 오차 패턴을 갖는 SVR — 앙상블 다양성 확보 목적."""
    preprocessor = make_preprocessor(train_features)
    train_matrix = preprocessor.fit_transform(train_features)
    val_matrix = preprocessor.transform(val_features)

    scaler = StandardScaler()
    train_scaled = scaler.fit_transform(train_matrix)
    val_scaled = scaler.transform(val_matrix)

    model = SVR(kernel="rbf", C=10.0, epsilon=0.01, gamma="scale")
    model.fit(train_scaled, target)
    return model.predict(val_scaled)


## 5-Fold CV — OOF 수집 + RidgeCV 스태킹

세 모델(Tree / Pair-kNN / SVR)의 OOF 예측을 모아 `RidgeCV`로 최적 가중치를 학습합니다.
V34처럼 quantile·weight를 수동으로 여러 버전(v1~v34) 반복 탐색하는 대신,
이 방식은 5-fold 안에서 한 번에 가중치를 데이터로부터 추정합니다.


In [5]:

DATA_DIR = Path("/Users/bitsaem/Desktop/stress_project/open") # 실제 경로로 수정
OUTPUT_DIR = Path("/Users/bitsaem/Desktop/stress_project/8_6/result")

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

if set(train.columns) - {TARGET} != set(test.columns):
    raise ValueError("Train과 Test의 입력 컬럼이 다릅니다.")
if train[TARGET].isna().any():
    raise ValueError("Train 타깃에 결측값이 있습니다.")

raw_X = train.drop(columns=[TARGET, ID_COLUMN]).reset_index(drop=True)
raw_X_test = test.drop(columns=[ID_COLUMN]).reset_index(drop=True)
y = train[TARGET].reset_index(drop=True)

cv = KFold(5, shuffle=True, random_state=RANDOM_SEED)
oof_tree = np.zeros(len(y))
oof_pair = np.zeros(len(y))
oof_svr = np.zeros(len(y))
fold_rows = []

for fold, (train_idx, val_idx) in enumerate(cv.split(raw_X), start=1):
    train_raw_fold = raw_X.iloc[train_idx].reset_index(drop=True)
    val_raw_fold = raw_X.iloc[val_idx].reset_index(drop=True)
    y_train_fold = y.iloc[train_idx].reset_index(drop=True)
    y_val_fold = y.iloc[val_idx].reset_index(drop=True)

    train_feat, val_feat = build_fold_features(train_raw_fold, val_raw_fold)

    pred_tree = tree_prediction(train_feat, val_feat, y_train_fold)
    pred_pair = pair_knn_prediction(train_feat, val_feat, y_train_fold)
    pred_svr = svr_prediction(train_feat, val_feat, y_train_fold)

    oof_tree[val_idx] = pred_tree
    oof_pair[val_idx] = pred_pair
    oof_svr[val_idx] = pred_svr

    fold_rows.append({
        "Fold": fold,
        "Tree_MAE": mean_absolute_error(y_val_fold, pred_tree),
        "Pair_MAE": mean_absolute_error(y_val_fold, pred_pair),
        "SVR_MAE": mean_absolute_error(y_val_fold, pred_svr),
    })
    print(f"Fold {fold} done")

fold_df = pd.DataFrame(fold_rows)
display(fold_df)
print("개별 모델 평균 MAE:")
print(fold_df[["Tree_MAE", "Pair_MAE", "SVR_MAE"]].mean())

stack_X = np.column_stack([oof_tree, oof_pair, oof_svr])
meta_model = RidgeCV(alphas=[0.001, 0.01, 0.1, 1.0, 10.0])
meta_model.fit(stack_X, y)
oof_stacked = meta_model.predict(stack_X)
oof_stacked_clipped = np.clip(np.round(oof_stacked, 2), 0.0, 1.0)

print("메타모델(RidgeCV) 계수 [tree, pair, svr]:", meta_model.coef_, "intercept:", meta_model.intercept_)
print("V35 Stacked CV MAE:", f"{mean_absolute_error(y, oof_stacked_clipped):.6f}")


Fold 1 done
Fold 2 done
Fold 3 done
Fold 4 done
Fold 5 done


,Fold,Tree_MAE,Pair_MAE,SVR_MAE
0,1,0.163707,0.224633,0.168306
1,2,0.179539,0.233254,0.191600
2,3,0.180209,0.232925,0.190775
3,4,0.194068,0.241534,0.206810
4,5,0.183514,0.233251,0.184666


개별 모델 평균 MAE:
Tree_MAE    0.180207
Pair_MAE    0.233119
SVR_MAE     0.188432
dtype: float64
메타모델(RidgeCV) 계수 [tree, pair, svr]: [ 1.40019072  0.10061519 -0.01082025] intercept: -0.2308305194914998
V35 Stacked CV MAE: 0.158550


## 최종 제출 파일 생성

전체 train 데이터로 세 모델을 다시 학습하고, 위에서 학습한 메타모델 가중치로 test를 예측합니다.


In [6]:

final_train_feat, final_test_feat = build_fold_features(raw_X, raw_X_test)

final_tree = tree_prediction(final_train_feat, final_test_feat, y)
final_pair = pair_knn_prediction(final_train_feat, final_test_feat, y)
final_svr = svr_prediction(final_train_feat, final_test_feat, y)

final_stack_X = np.column_stack([final_tree, final_pair, final_svr])
final_prediction = np.clip(np.round(meta_model.predict(final_stack_X), 2), 0.0, 1.0)

output = submission.copy()
output[TARGET] = final_prediction
submit_path = OUTPUT_DIR / "submit_v35_stack_svr_0806.csv"
output.to_csv(submit_path, index=False, encoding="utf-8")

print("Saved:", submit_path)
print("Prediction range:", final_prediction.min(), final_prediction.max())
display(output.head(10))


Saved: /Users/bitsaem/Desktop/stress_project/8_6/result/submit_v35_stack_svr_0806.csv
Prediction range: 0.0 1.0


,ID,stress_score
0,TEST_0000,0.52
1,TEST_0001,0.95
2,TEST_0002,0.23
3,TEST_0003,0.47
4,TEST_0004,0.52
5,TEST_0005,0.49
6,TEST_0006,0.76
7,TEST_0007,0.49
8,TEST_0008,0.51
9,TEST_0009,0.83
